In [2]:
import pandas as pd
!pip install great_expectations

import great_expectations as gx
context = gx.get_context()

In [3]:
df=pd.read_csv("/content/student_sleep_screen_time_academic_performance.csv")

In [ ]:
df.head(2)

,student_id,age,gender,year_of_study,major_category,residence_type,chronotype,avg_bedtime_hour,avg_sleep_hours_weekday,weekend_sleep_delta_hours,...,naps_per_week,sleep_quality_score,exercise_days_per_week,energy_level,concentration_difficulty,weekly_study_hours,class_attendance_percent,has_part_time_job,semester_type,cgpa
0,STU2001,21,Male,3,Humanities,With Family,Early Bird,22.3,7.6,2.0,...,0,10.0,7,5.0,7.0,15.6,80.0,No,Spring,2.99
1,STU2002,20,Female,2,STEM,Dormitory,Intermediate,23.1,5.9,2.7,...,3,7.0,3,6.0,7.0,17.5,78.9,No,Summer,2.88


In [ ]:
df.isnull().sum()

,0
student_id,0
age,0
gender,0
year_of_study,0
major_category,0
residence_type,0
chronotype,0
avg_bedtime_hour,0
avg_sleep_hours_weekday,0
weekend_sleep_delta_hours,0


In [ ]:
#Check the number of rows and columns
print("\nDataset Shape:")
print(df.shape)

#Check column names and data types
print("\nDataset Information:")
df.info()

#Count missing values in each column
print("\nMissing Values:")
print(df.isnull().sum())

# Count duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())

# Generate summary statistics
print("\nSummary Statistics:")
print(df.describe(include="all"))

# Check column names
print("\nColumn Names:")
print(df.columns.tolist())


Dataset Shape:
(500, 25)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   student_id                 500 non-null    object 
 1   age                        500 non-null    int64  
 2   gender                     500 non-null    object 
 3   year_of_study              500 non-null    int64  
 4   major_category             500 non-null    object 
 5   residence_type             500 non-null    object 
 6   chronotype                 500 non-null    object 
 7   avg_bedtime_hour           500 non-null    float64
 8   avg_sleep_hours_weekday    500 non-null    float64
 9   weekend_sleep_delta_hours  500 non-null    float64
 10  pre_sleep_screen_time_min  500 non-null    int64  
 11  daily_screen_time_hours    500 non-null    float64
 12  primary_device             500 non-null    object 
 13  de

# Scoring all 6 DAMA dimension

In [ ]:
# Calculate data quality metrics
total_rows = len(df)
total_cells = df.size

# Completeness: percentage of non-missing cells
completeness = (
    100 * (1 - df.isnull().sum().sum() / total_cells)
    if total_cells else 100
)

# Uniqueness: percentage of rows that are not duplicates
uniqueness = (
    100 * (1 - df.duplicated().sum() / total_rows)
    if total_rows else 100
)

# Validity: percentage of cells that are non-null
# and have no obvious blank strings.
nonblank = df.astype("string").apply(
    lambda col: col.str.strip().ne("").fillna(False)
)
validity = 100 * nonblank.to_numpy().sum() / total_cells if total_cells else 100

#Consistency: placeholder score until business rules are defined
consistency = None

#Accuracy: requires comparison with a trusted source
accuracy = None

#Timeliness: requires a date column and freshness requirement
timeliness = None

# Display results
scores = pd.DataFrame({
    "Dimension": [
        "Completeness", "Uniqueness", "Validity",
        "Consistency", "Accuracy", "Timeliness"
    ],
    "Score (%)": [
        round(completeness, 2),
        round(uniqueness, 2),
        validity,
        consistency,
        accuracy,
        timeliness
    ]
})

print(scores)

      Dimension  Score (%)
0  Completeness     99.590
1    Uniqueness    100.000
2      Validity     99.592
3   Consistency        NaN
4      Accuracy        NaN
5    Timeliness        NaN


# Finding out the data quality using great_expections

In [27]:
import great_expectations as gx
context = gx.get_context()

# Connect to your DataFrame
source = context.data_sources.add_pandas("student_source")
asset = source.add_dataframe_asset(name="students")
batch_def = asset.add_batch_definition_whole_dataframe("student_batch")

batch = batch_def.get_batch(batch_parameters={"dataframe": df})

# Validate student ID
r1 = batch.validate(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="student_id")
)

# Validate age
r2 = batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="age", min_value=16, max_value=100
    )
)

# Validate CGPA
r3 = batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="cgpa", min_value=0, max_value=4
    )
)

# Sleep hours must be between 0 and 24
r4 = batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="avg_sleep_hours_weekday", min_value=0, max_value=24
    )
)

# Daily screen time must be between 0 and 24
r5 = batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="daily_screen_time_hours", min_value=0, max_value=24
    )
)

# Attendance must be between 0 and 100
r6 = batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="class_attendance_percent", min_value=0, max_value=100
    )
)

#Sleep quality must be between 1 to 10
r7 = batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="sleep_quality_score", min_value=1, max_value=10
    )
)

#Ensure that attendence level is between 50 and 100
r8 = batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="class_attendance_percent", min_value=50, max_value=100
    )
)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpuie4dpxa' for ephemeral docs site


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

##The expectation has found out the bad data

In [31]:
# Create a copy of the original DataFrame
bad_df = df.copy()

# Error 1: Invalid age
bad_df.loc[0, "age"] = 200

# Error 2: Duplicate student ID
bad_df.loc[1, "student_id"] = bad_df.loc[0, "student_id"]

# Error 3: Invalid CGPA
bad_df.loc[2, "cgpa"] = 5.5

# Connect the incorrect DataFrame to Great Expectations
bad_batch = batch_def.get_batch(
    batch_parameters={"dataframe": bad_df}
)

# Check whether GX catches the three errors

# 1. Validate age
r1 = bad_batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="age", min_value=16, max_value=100
    )
)

# 2. Validate duplicate student ID
r2 = bad_batch.validate(
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="student_id"
    )
)

# 3. Validate CGPA
r3 = bad_batch.validate(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="cgpa", min_value=0, max_value=4
    )
)

# Display the results
print("Invalid age detected:", not r1.success)
print("Duplicate student ID detected:", not r2.success)
print("Invalid CGPA detected:", not r3.success)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Invalid age detected: True
Duplicate student ID detected: True
Invalid CGPA detected: True
